##customers

In [0]:
customers_bronze = spark.table(
    "databricks_incremental_medallion_lakehouse.bronze.customers_bronze"
)

##orders

In [0]:
orders_bronze = spark.table(
    "databricks_incremental_medallion_lakehouse.bronze.orders_bronze"
)

##products

In [0]:
products_bronze = spark.table(
    "databricks_incremental_medallion_lakehouse.bronze.products_bronze"
)

#clean customer table


In [0]:
from pyspark.sql import functions as F

In [0]:
customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("customer_name").isNotNull())
)

#silver data clean

In [0]:
orders_silver = (
    orders_bronze
    .dropDuplicates(["order_id"])
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("amount") >= 0)
)

#clean products table

In [0]:
products_silver = (
    products_bronze
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("price") >= 0)
)

In [0]:
customers_silver.printSchema()

In [0]:
%sql
-- create table databricks_incremental_medallion_lakehouse.silver.customers_silver as select * from databricks_incremental_medallion_lakehouse.bronze.customers_bronze
-- where 1=2; 

In [0]:
%sql
-- create table databricks_incremental_medallion_lakehouse.silver.orders_silver as select * -- from databricks_incremental_medallion_lakehouse.bronze.orders_bronze
-- where 1=2; 

In [0]:
%sql
-- create table databricks_incremental_medallion_lakehouse.silver.products_silver as select * from databricks_incremental_medallion_lakehouse.bronze.products_bronze
-- where 1=2; 

#overwrite the data to silver tables


In [0]:
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "databricks_incremental_medallion_lakehouse.silver.customers_silver"
    )

In [0]:
orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "databricks_incremental_medallion_lakehouse.silver.orders_silver"
    )

In [0]:
products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "databricks_incremental_medallion_lakehouse.silver.products_silver"
    )